# Digital Image Processing Lab - Assignment 4
**Problem Statements:**
1. **WAP to enhance the input image using Log transform.** Assume $c = 1$.
2. **WAP to enhance the input image using Power law / gamma transform.** Assume $c = 1$, and input the value of gamma from the user.
3. **WAP to enhance the input image using Intensity level slicing (Contrast stretching).**
   - The input range of intensity values and the desired range of intensity values in the output image are obtained from the user.
   *(Example: The input range of intensity values from 80 to 120 is stretched to 50-150 in the output image).*

---
### Mathematical Formulation
1. **Log Transformation:**
   $$s = c \cdot \ln(1 + r)$$
   Log transform compresses the dynamic range of images with large variations in pixel values (e.g. Fourier spectrum). When $c=1$, values $r \in [0, 255]$ map to $[0, \ln(256)] \approx [0, 5.545]$. To display on standard 8-bit displays, the scale factor is normalized as:
   $$s_{\text{display}} = \frac{255}{\ln(1 + \max(r))} \ln(1 + r)$$

2. **Power Law (Gamma) Transformation:**
   $$s = c \cdot r^\gamma$$
   For normalized input $r \in [0, 1]$:
   - $\gamma < 1$: Expands dark pixels, brightens image (enhances underexposed images).
   - $\gamma > 1$: Compresses dark pixels, darkens image (enhances washed out/overexposed images).

3. **Contrast Stretching (Piecewise Linear Transformation):**
   Maps an input intensity range $[r_1, r_2]$ to an output intensity range $[s_1, s_2]$:
   $$s = \begin{cases}
   \frac{s_1}{r_1} r & 0 \le r < r_1 \\
   \frac{s_2 - s_1}{r_2 - r_1}(r - r_1) + s_1 & r_1 \le r \le r_2 \\
   \frac{255 - s_2}{255 - r_2}(r - r_2) + s_2 & r_2 < r \le 255
   \end{cases}$$


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# Cell 2: Test Low-Contrast & High-Dynamic-Range Image
def generate_low_contrast_image():
    """Generates a low-contrast image having values concentrated in a narrow band [80, 120]."""
    x = np.linspace(-2, 2, 256)
    y = np.linspace(-2, 2, 256)
    xx, yy = np.meshgrid(x, y)
    pattern = np.sin(xx**2 + yy**2)
    # Narrow range: 80 to 120
    narrow = (pattern * 20 + 100).astype(np.uint8)
    cv2.putText(narrow, "DIP 2026", (40, 140), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 118, 3)
    return narrow

img = generate_low_contrast_image()
print(f"Test Image Created. Min={img.min()}, Max={img.max()}")


In [ ]:
# Cell 3: Task 1 - Log Transformation (c = 1)
def log_transform(image, c=1.0, scale_to_255=True):
    """Applies s = c * log(1 + r)."""
    # Convert image to float
    r = image.astype(np.float64)
    s_raw = c * np.log(1.0 + r)
    
    if scale_to_255:
        # Scale to standard 8-bit dynamic range for visualization
        c_norm = 255.0 / np.log(1.0 + np.max(r))
        s_display = c_norm * np.log(1.0 + r)
        return s_raw, np.clip(np.round(s_display), 0, 255).astype(np.uint8)
    return s_raw, s_raw

s_raw_log, s_display_log = log_transform(img, c=1.0)
print(f"Log Transform (c=1.0): raw range [{s_raw_log.min():.2f}, {s_raw_log.max():.2f}], display range [{s_display_log.min()}, {s_display_log.max()}]")


In [ ]:
# Cell 4: Task 2 - Power Law / Gamma Transform (c = 1)
gamma_val = 0.5 #@param {type:"number"}

def power_law_transform(image, gamma, c=1.0):
    """Applies s = c * r^gamma on normalized image [0, 1]."""
    # Normalize to [0, 1]
    norm_img = image.astype(np.float64) / 255.0
    s = c * (norm_img ** gamma)
    s_scaled = np.clip(s * 255.0, 0, 255).astype(np.uint8)
    return s_scaled

gamma_enhanced = power_law_transform(img, gamma=gamma_val, c=1.0)
print(f"Gamma ({gamma_val}) Transform complete. Output range: [{gamma_enhanced.min()}, {gamma_enhanced.max()}]")


In [ ]:
# Cell 5: Task 3 - Intensity Level Slicing / Contrast Stretching
# User Defined Ranges (as in example: [80, 120] -> [50, 150])
r1 = 80 #@param {type:"integer"}
r2 = 120 #@param {type:"integer"}
s1 = 50 #@param {type:"integer"}
s2 = 150 #@param {type:"integer"}

def contrast_stretch(image, r1, r2, s1, s2):
    """Piecewise linear contrast stretching from [r1, r2] to [s1, s2]."""
    r = image.astype(np.float64)
    s = np.zeros_like(r)
    
    # Region 1: 0 <= r < r1
    mask1 = (r < r1)
    s[mask1] = (s1 / r1) * r[mask1] if r1 > 0 else 0
    
    # Region 2: r1 <= r <= r2
    mask2 = (r >= r1) & (r <= r2)
    s[mask2] = ((s2 - s1) / (r2 - r1)) * (r[mask2] - r1) + s1
    
    # Region 3: r > r2
    mask3 = (r > r2)
    s[mask3] = ((255 - s2) / (255 - r2)) * (r[mask3] - r2) + s2 if r2 < 255 else s2
    
    return np.clip(np.round(s), 0, 255).astype(np.uint8)

stretched_img = contrast_stretch(img, r1, r2, s1, s2)
print(f"Contrast stretched [{r1}, {r2}] -> [{s1}, {s2}]. Output range: [{stretched_img.min()}, {stretched_img.max()}]")


In [ ]:
# Cell 6: Visualizations of All Three Transformations
fig, axes = plt.subplots(3, 3, figsize=(16, 14))

# Transfer Function Curves
r_vals = np.arange(256)

# Row 1: Log Transform
axes[0, 0].plot(r_vals, np.log(1 + r_vals), 'b-', lw=2, label='c=1 raw')
axes[0, 0].set_title("Log Transform Transfer Function (c=1)", fontweight='bold')
axes[0, 0].set_xlabel("Input r"); axes[0, 0].set_ylabel("Output s")
axes[0, 0].grid(True, alpha=0.3); axes[0, 0].legend()

axes[0, 1].imshow(img)
axes[0, 1].set_title("Original Low-Contrast Image", fontweight='bold'); axes[0, 1].axis('off')

axes[0, 2].imshow(s_display_log)
axes[0, 2].set_title("Enhanced: Log Transform", fontweight='bold'); axes[0, 2].axis('off')

# Row 2: Gamma Transform
for g in [0.4, 0.7, 1.0, 1.5, 2.5]:
    axes[1, 0].plot(r_vals, 255 * ((r_vals/255.0)**g), label=f'gamma={g}')
axes[1, 0].set_title("Power-Law (Gamma) Family (c=1)", fontweight='bold')
axes[1, 0].set_xlabel("Input r"); axes[1, 0].set_ylabel("Output s")
axes[1, 0].grid(True, alpha=0.3); axes[1, 0].legend()

axes[1, 1].imshow(img)
axes[1, 1].set_title("Original Image", fontweight='bold'); axes[1, 1].axis('off')

axes[1, 2].imshow(gamma_enhanced)
axes[1, 2].set_title(f"Enhanced: Gamma (gamma={gamma_val})", fontweight='bold'); axes[1, 2].axis('off')

# Row 3: Contrast Stretching
s_curve = contrast_stretch(r_vals, r1, r2, s1, s2)
axes[2, 0].plot(r_vals, s_curve, 'r-', lw=2.5, label='Contrast Stretch')
axes[2, 0].axvline(r1, color='gray', linestyle='--'); axes[2, 0].axvline(r2, color='gray', linestyle='--')
axes[2, 0].axhline(s1, color='gray', linestyle=':'); axes[2, 0].axhline(s2, color='gray', linestyle=':')
axes[2, 0].set_title(f"Contrast Stretch Function [{r1},{r2}]->[{s1},{s2}]", fontweight='bold')
axes[2, 0].set_xlabel("Input r"); axes[2, 0].set_ylabel("Output s")
axes[2, 0].grid(True, alpha=0.3); axes[2, 0].legend()

axes[2, 1].hist(img.ravel(), bins=256, range=[0, 256], color='gray', alpha=0.7)
axes[2, 1].set_title("Histogram: Original (Narrow)", fontweight='bold')

axes[2, 2].hist(stretched_img.ravel(), bins=256, range=[0, 256], color='teal', alpha=0.7)
axes[2, 2].set_title(f"Histogram: Stretched [{s1}, {s2}]", fontweight='bold')

plt.tight_layout()
plt.show()


### Summary of Enhancements
1. **Log Transform:** Effectively expands low-intensity dark areas while compressing very high intensity areas.
2. **Power Law (Gamma):** Flexible parametric curve enabling both brightening ($\gamma < 1$) and darkening ($\gamma > 1$).
3. **Contrast Stretching:** Stretches the narrow dynamic range $[80, 120]$ to $[50, 150]$, significantly enhancing visibility and contrast of subtle features.
